<a href="https://colab.research.google.com/github/DivineUI/FairLoan/blob/main/data_cleaning_and_encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Data Cleaning & Missing Values



In [1]:

from google.colab import files
uploaded = files.upload()

Saving loan_data.csv to loan_data.csv


In [2]:
import pandas as pd
import numpy as np

# STEP 1: Load the raw data
df = pd.read_csv('loan_data.csv')
print("Original shape:", df.shape)

Original shape: (45000, 14)


In [3]:
# STEP 2: Check for missing values
print(df.isnull().sum())


person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64


#### No missing values were found

In [4]:
# STEP 3: Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())


Duplicate rows: 0


In [5]:
# STEP 4: Check numeric columns for outliers
num_cols = ['person_age','person_income','person_emp_exp','loan_amnt',
            'loan_int_rate','loan_percent_income',
            'cb_person_cred_hist_length','credit_score']
print(df[num_cols].describe())

         person_age  person_income  person_emp_exp     loan_amnt  \
count  45000.000000   4.500000e+04    45000.000000  45000.000000   
mean      27.764178   8.031905e+04        5.410333   9583.157556   
std        6.045108   8.042250e+04        6.063532   6314.886691   
min       20.000000   8.000000e+03        0.000000    500.000000   
25%       24.000000   4.720400e+04        1.000000   5000.000000   
50%       26.000000   6.704800e+04        4.000000   8000.000000   
75%       30.000000   9.578925e+04        8.000000  12237.250000   
max      144.000000   7.200766e+06      125.000000  35000.000000   

       loan_int_rate  loan_percent_income  cb_person_cred_hist_length  \
count   45000.000000         45000.000000                45000.000000   
mean       11.006606             0.139725                    5.867489   
std         2.978808             0.087212                    3.879702   
min         5.420000             0.000000                    2.000000   
25%         8.590000  

In [6]:
# STEP 5: Find impossible values
# Catches absurd single values AND absurd combinations
# (e.g. more work experience than physically possible, assuming legal working age of 18)
outliers = df[
    (df['person_age'] > 100) |
    (df['person_emp_exp'] > 60) |
    (df['person_emp_exp'] > (df['person_age'] - 18))
]
print("Rows with impossible age/experience:", len(outliers))
print(outliers[['person_age','person_emp_exp','person_income']])

Rows with impossible age/experience: 10
       person_age  person_emp_exp  person_income
81          144.0             125       300616.0
183         144.0             121       241424.0
575         123.0             101        97140.0
747         123.0             100        94723.0
32297       144.0             124      7200766.0
32416        94.0              76        29738.0
32422        80.0              62        77894.0
32506        84.0              61       114705.0
37930       116.0              93      5545545.0
38113       109.0              85      5556399.0


In [7]:
# STEP 6: Drop the impossible rows
# We drop instead of impute, because these are clearly data entry errors
# (e.g. age 144, 125 years of work experience), not real extreme cases.
df_cleaned = df[
    (df['person_age'] <= 100) &
    (df['person_emp_exp'] <= 60) &
    (df['person_emp_exp'] <= (df['person_age'] - 18))
].copy()
print("New shape after dropping outliers:", df_cleaned.shape)

New shape after dropping outliers: (44990, 14)


In [8]:
# STEP 7: Save the cleaned dataset
df_cleaned.to_csv('credit_cleaned.csv', index=False)
print("Saved as credit_cleaned.csv")

Saved as credit_cleaned.csv


In [9]:
# STEP 8 (optional): Download the cleaned file to your computer
from google.colab import files
files.download('credit_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Data Cleaning Notes

- Checked for missing values: none found.
- Checked for duplicate rows: none found.
- Found 10 rows with impossible values (age up to 144, employment experience up to 125 years). Also checked whether experience makes sense relative to age (assuming legal working age of 18) — no additional rows were caught by this rule, but it is included for robustness. These are data entry errors, not real extreme cases, so they were dropped rather than imputed.
- Income has some high values (up to ~$7.2M) but these are spread across different rows and look like real high-income applicants, so we kept them as is.
- Final cleaned dataset: 44,990 rows (down from 45,000).

In [10]:
from google.colab import files
uploaded = files.upload()

Saving credit_cleaned (2).csv to credit_cleaned (2).csv


In [12]:
import pandas as pd

df = pd.read_csv('credit_cleaned (2).csv')

# Binary encoding
df['person_gender'] = df['person_gender'].map({'female': 0, 'male': 1})
df['previous_loan_defaults_on_file'] = df['previous_loan_defaults_on_file'].map({'No': 0, 'Yes': 1})

# Ordinal encoding for education (natural order)
education_order = {'High School': 0, 'Associate': 1, 'Bachelor': 2, 'Master': 3, 'Doctorate': 4}
df['person_education'] = df['person_education'].map(education_order)

# One-hot encoding for home_ownership and loan_intent (no natural order)
df = pd.get_dummies(df, columns=['person_home_ownership', 'loan_intent'], prefix=['home', 'intent'])

# Convert one-hot boolean columns to 0/1
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

df.to_csv('credit_encoded(2).csv', index=False)
print(df.shape)
df.head()

(44990, 22)


,person_age,person_gender,person_education,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,...,home_MORTGAGE,home_OTHER,home_OWN,home_RENT,intent_DEBTCONSOLIDATION,intent_EDUCATION,intent_HOMEIMPROVEMENT,intent_MEDICAL,intent_PERSONAL,intent_VENTURE
0,22.0,0,3,71948.0,0,35000.0,16.02,0.49,3.0,561,...,0,0,0,1,0,0,0,0,1,0
1,21.0,0,0,12282.0,0,1000.0,11.14,0.08,2.0,504,...,0,0,1,0,0,1,0,0,0,0
2,25.0,0,0,12438.0,3,5500.0,12.87,0.44,3.0,635,...,1,0,0,0,0,0,0,1,0,0
3,23.0,0,2,79753.0,0,35000.0,15.23,0.44,2.0,675,...,0,0,0,1,0,0,0,1,0,0
4,24.0,1,3,66135.0,1,35000.0,14.27,0.53,4.0,586,...,0,0,0,1,0,0,0,1,0,0


**Encoding Notes**

- Started with the cleaned dataset: 44,990 rows, 14 columns.
- Gender and previous defaults: turned into 0s and 1s, since each only has two options.
- Education: turned into numbers 0 through 4, in order from High School to Doctorate, so the ranking is preserved.
- Home ownership and loan intent: split into separate yes/no columns for each category, since there's no natural order between them.
- Loan status (our target) was left as is.
- Final dataset: 44,990 rows, 22 columns, the extra columns coming from splitting home ownership and loan intent.